# Recipes · Interpretable concept retrieval

Search a small recipe collection with binarized UCE concepts. Each result shows the **shared concept labels** contributing to an IDF-weighted retrieval score.

## Install ConceptExplorer from GitHub

Set `GITHUB_REPO` to your published repository (and `PACKAGE_SUBDIR` to the folder containing `pyproject.toml` if the package is in a monorepo). Use a release tag instead of `main` for stable results.


In [ ]:
import sys, subprocess

GITHUB_REPO = "YOUR_ORG/YOUR_REPO"  # Change to the actual GitHub repository
GIT_REF = "main"                  # Or a tagged release
PACKAGE_SUBDIR = ""               # Example: "packages/uce-concept-explorer"

if GITHUB_REPO == "YOUR_ORG/YOUR_REPO":
    raise ValueError("Set GITHUB_REPO to the repository hosting uce-concept-explorer")

url = f"uce-concept-explorer @ git+https://github.com/{GITHUB_REPO}.git@{GIT_REF}"
if PACKAGE_SUBDIR:
    url += f"#subdirectory={PACKAGE_SUBDIR}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", url])
from uce_concept_explorer import ConceptExplorer, load_metadata


## 1. Install and load UCE-Gemma

Select **Runtime → Change runtime type → T4 GPU** in Colab. This notebook downloads the model weights only for encoding. Subsequent analysis is CPU-only. The Hugging Face base model may require an access token in Colab Secrets (`HF_TOKEN`).

In [ ]:
%pip -q install -U huggingface_hub datasets scipy pandas scikit-learn matplotlib wordcloud
from huggingface_hub import snapshot_download
from pathlib import Path
MODEL_ID = "klpeng/uce-gemma-80k"
model_dir = snapshot_download(MODEL_ID)
import subprocess, sys
req = Path(model_dir) / "requirements.txt"
if req.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

In [ ]:
import os, sys, torch, numpy as np
from scipy import sparse
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token: os.environ['HF_TOKEN'] = token
except Exception: pass
sys.path.insert(0, model_dir)
from uce_gemma import UCEGemma
model = UCEGemma.from_pretrained(model_dir, device='cuda' if torch.cuda.is_available() else 'cpu')
print('Model loaded.')

## Load 1,000 recipes

In [ ]:
from datasets import load_dataset

ds = load_dataset('LIUH0115/RecipeNLG_Group8', split='train', streaming=True)
recipes = list(ds.shuffle(seed=42,buffer_size=10000).take(1000))
print('Fields:', list(recipes[0]))
def as_text(value):
    if isinstance(value, (list,tuple)):
        return ', '.join(map(str,value))
    return str(value or '')
texts = [f"{r.get('title','')}\nIngredients: {as_text(r.get('ingredients'))}\nDirections: {as_text(r.get('directions'))}" for r in recipes]

## 2. Encode once

Batch size is deliberately small for a 16 GB T4. Cache the sparse scores to avoid rerunning the GPU. Documents are encoded with the same ordering as `texts`.

In [ ]:
THRESHOLD = 0.92
BATCH_SIZE = 2
chunks = []
for i in range(0, len(texts), BATCH_SIZE):
    with torch.inference_mode():
        chunk = model.encode(list(texts[i:i+BATCH_SIZE]))
    chunks.append(sparse.csr_matrix(chunk))
    if i % 50 == 0: print(f'{i}/{len(texts)}', end='\r')
X = sparse.vstack(chunks, format='csr')
sparse.save_npz('uce_scores.npz', X)
print('Scores:', X.shape, 'nnz:', X.nnz)
# Later sessions: X = sparse.load_npz('uce_scores.npz')

## Explore concepts (CPU only)

The implementation is imported from GitHub; only dataset-specific analysis appears here. The binary threshold applies to all analyses.


In [ ]:
# Metadata-only loader: no weights are downloaded by this call.
metadata = load_metadata(MODEL_ID)
E = ConceptExplorer(X, texts, metadata, threshold=THRESHOLD)
print('Encoded texts:', len(texts), 'active features:', int((E.counts>0).sum()))
display(E.prevalent(top_n=15))

**Optional fidelity filtering.** The UCE model card describes F1 based on up to 30 positive examples and 30 sparse zero examples per concept; it is an *automated judgment* metric, not corpus-calibrated precision or recall.

In [ ]:
print('Fidelity columns:', [c for c in E.table.columns if c in ('f1','precision','recall','accuracy')])
# Uncomment if available:
# E.set_fidelity_filter(min_fidelity=0.8, field='f1')
# display(E.prevalent(top_n=10))

## IDF-weighted binary concept retrieval

In [ ]:
def search_recipes(query, n=5, n_concepts=5):
    q_raw = sparse.csr_matrix(model.encode([query]))
    q = (q_raw > THRESHOLD).astype(float).tocsr()
    df = E.counts
    idf = np.log((len(texts)+1)/(df+1))
    weighted_q = q.multiply(idf).tocsr()
    scores = (E.Z @ weighted_q.T).toarray().ravel()
    top_ids = np.argsort(-scores)[:n]
    for i in top_ids:
        if scores[i]<=0: continue
        terms = E.Z.getrow(i).multiply(weighted_q).tocsr()
        idx = np.argsort(-terms.data)[:n_concepts]
        labels = [(E.name(terms.indices[j]),float(terms.data[j])) for j in idx]
        print(f"\n{recipes[i].get('title', 'Recipe')} | score={scores[i]:.3f}")
        print('Ingredients:', as_text(recipes[i].get('ingredients'))[:260])
        print('Contributing concepts:', labels)

search_recipes('spicy Asian noodles', n=5)

## Optional regression for numeric targets

If your recipe sample has a continuous target (e.g. preparation time), pass an aligned numeric array to `E.regressor(y, n_concepts=5)`.
